In [4]:
!pip install triton -q

In [9]:
!pip show triton

Name: triton
Version: 3.6.0
Summary: A language and compiler for custom Deep Learning operations
Home-page: https://github.com/triton-lang/triton/
Author: Philippe Tillet
Author-email: phil@openai.com
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: torch


In [2]:
import torch
import triton
import triton.language as tl

device = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device)
name = props.name
vram = props.total_memory / 1e9
capability = f"{props.major}.{props.minor}"

print(f"--- HARDWARE REPORT ---")
print(f"GPU: {name}")
print(f"VRAM: {vram:.2f} GB")
print(f"Compute Capability: {capability}")
print(f"Triton Version: {triton.__version__}")

--- HARDWARE REPORT ---
GPU: Tesla T4
VRAM: 15.64 GB
Compute Capability: 7.5
Triton Version: 3.6.0


In [3]:
@triton.jit
def add_kernel(x_ptr, y_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)

    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)

    output = x + y

    tl.store(output_ptr + offsets, output, mask=mask)

def triton_add(x: torch.Tensor, y: torch.Tensor):
    output = torch.empty_like(x)
    n_elements = output.numel()

    # Grid: How many 'Programs' (thread blocks) do we launch?
    grid = lambda meta: (triton.cdiv(n_elements, meta['BLOCK_SIZE']),)

    # Launch kernel
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output

size = 100_000
x = torch.rand(size, device='cuda')
y = torch.rand(size, device='cuda')
z = triton_add(x, y)

torch.testing.assert_close(z, x + y)
print("Triton Vector Add: SUCCESS")

Triton Vector Add: SUCCESS


## Baseline Exploration

In [ ]:
import torch
import torch.nn.functional as fn


In [12]:
import triton

def pytorch_attention_baseline(q, k, v, softmax_scale):
    # compute scaled attention scores = (q.k) * softmax_scale 
    # shape (B, H, S, S)
    scores = torch.matmul(q, k.transpose(-2, -1)) * softmax_scale

    # probs of score = shape((B, H, S, S)
    probs = fn.softmax(scores, dim=-1)

    # weighted sumshape(B, H, S, D)
    output = torch.matmul(probs, v)

    return output

B, H, S, D = 4, 32, 1024, 64
sm_scale = 1.0 / (D ** 0.5)

q = torch.randn((B, H, S, D), device='cuda', dtype=torch.float16)
k = torch.randn((B, H, S, D), device='cuda', dtype=torch.float16)
v = torch.randn((B, H, S, D), device='cuda', dtype=torch.float16)

baseline_result = pytorch_attention_baseline(q, k, v, sm_scale)
print(f"Baseline Output Shape: {baseline_result.shape}")

Baseline Output Shape: torch.Size([4, 32, 1024, 64])


## T4 specs - 
 - performance = 65 TFLOPS FP16.
 - Memory Bandwidth: 320+ GB/s.

### Baseline observation :- The T4 can do at 320 gb/s, but baseline working at 37.18 GB/s which is only ~10% of potential

In [13]:
def benchmark_baseline():
    # Warmup
    for _ in range(10):
        pytorch_attention_baseline(q, k, v, sm_scale)
    
    # Synchronize ensures the GPU is actually finished before we start the timer
    torch.cuda.synchronize()
    
    # triton.testing.do_bench handles the heavy lifting of timing
    ms = triton.testing.do_bench(lambda: pytorch_attention_baseline(q, k, v, sm_scale))
    
    # Calculate Bandwidth (How much data did we move?)
    # Q, K, V, and Output are read/written.
    # We also created a massive intermediate 'scores' matrix.
    total_bytes = (q.numel() + k.numel() + v.numel() + (S*S*B*H) + baseline_result.numel()) * 2 # 2 bytes for float16
    gb_per_sec = (total_bytes / 1e9) / (ms / 1e3)
    
    print(f"PyTorch Baseline Latency: {ms:.4f} ms")
    print(f"Estimated Bandwidth Used: {gb_per_sec:.2f} GB/s")

benchmark_baseline()

PyTorch Baseline Latency: 9.0239 ms
Estimated Bandwidth Used: 37.18 GB/s
